# Bronze layer - Data Ingestion Layer
# Objective
    - To ingest raw data directly from the Unity Catalog Volumn to Delta Table
# Responsibilities
    - This layer will ingest raw data from Unity Catalog
    - We will not add schema enforement, we will ingest the data as it is and since we are not defining the scehma it will be in String Format
    - We are adding technical metadata like source of the raw data and when it was ingested

In [0]:
# ------------------------
# Project Configuration
# ------------------------

CATALOG = "retail_demo"
RAW_SCHEMA = "raw_data"
BRONZE_SCHEMA = "bronze"
VOLUME = "olist_files"
BASE_PATH = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/{VOLUME}"


In [0]:
from pyspark.sql.functions import(
    current_timestamp,
    input_file_name
)

In [0]:
customers_df = (
    spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(f"{BASE_PATH}/olist_customers_dataset.csv")
)

In [0]:

# Displays the table read and limits to 10 rows; for now I will be commenting this cell
# display(customers_df.limit(10))


In [0]:
# Prints schema for the customer_df that we created; for now I will be commenting this cell
# customers_df.printSchema()

In [0]:
# We will add some metadata to not lose the info on when this data was added to delta lake and which source file this data come from
from pyspark.sql.functions import col


customers_bronze_df = (
    customers_df
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
)


In [0]:
# Display function; will be commenting this cell
# display(customers_bronze_df)

In [0]:
# Write function to write the customers_bronze_df to delta lake called customers_raw
(
    customers_bronze_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.customers_raw")
)
# format --> delta we will keep this format which will enable us in ACID transactions, versioning, timetravel and other features
# mode is overwrite, everytime we write this it will create new file
# saveAsTable creates a managed table inside unity catalog



In [0]:
# display function and I think this is native spark and not the pyspark version
display(
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers_raw").limit(10)
)

In [0]:
# spark native ig
spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.customers_raw"
).printSchema()

In [0]:
# -------------------------------
# Bronze ingestion function
# -------------------------------

# -----------------------------------
# Bronze Ingestion Function
# -----------------------------------

from pyspark.sql.functions import current_timestamp, input_file_name
from pyspark.sql.functions import col

def ingest_to_bronze(
    source_file: str,
    table_name: str
):
    """
    Reads a raw CSV file from the Unity Catalog Volume
    and ingests it into the Bronze layer.
    """

    print(f"Ingesting {source_file}...")

    df = (
        spark.read
            .option("header", True)
            .option("inferSchema", False)
            .csv(f"{BASE_PATH}/{source_file}")
    )

    df = (
        df
            .withColumn("ingestion_timestamp", current_timestamp())
            .withColumn("source_file", col("_metadata.file_path"))
    )

    (
        df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}")
    )

    print(f"✓ Created Bronze table: {table_name}")

In [0]:
ingest_to_bronze(
    source_file="olist_customers_dataset.csv",
    table_name="customers_raw"
)

In [0]:
# validate if the function successfully created the table
display(
    spark.table("retail_demo.bronze.customers_raw")
)

In [0]:
# adding orders table
ingest_to_bronze(
    "olist_orders_dataset.csv",
    "orders_raw"
)

In [0]:
# all at once now via the for loop
datasets = [
    ("olist_customers_dataset.csv", "customers_raw"),
    ("olist_orders_dataset.csv", "orders_raw"),
    ("olist_order_items_dataset.csv", "order_items_raw"),
    ("olist_products_dataset.csv", "products_raw"),
    ("olist_order_payments_dataset.csv", "payments_raw"),
    ("olist_order_reviews_dataset.csv", "reviews_raw"),
    ("olist_sellers_dataset.csv", "sellers_raw"),
    ("olist_geolocation_dataset.csv", "geolocation_raw"),
    ("product_category_name_translation.csv", "product_category_translation_raw")
]

for source_file, table_name in datasets:
    ingest_to_bronze(source_file, table_name)

In [0]:
# final ingestion with stats

# -----------------------------------
# Bronze Ingestion Function
# -----------------------------------

from datetime import datetime
from pyspark.sql.functions import current_timestamp, col


def ingest_to_bronze(
    source_file: str,
    table_name: str
):
    """
    Reads a raw CSV file from the Unity Catalog Volume
    and ingests it into the Bronze layer.

    Returns:
        dict: Ingestion statistics
    """

    print(f"\nIngesting {source_file}...")

    # Read raw CSV
    df = (
        spark.read
            .option("header", True)
            .option("inferSchema", False)
            .csv(f"{BASE_PATH}/{source_file}")
    )

    # Capture ingestion statistics
    row_count = df.count()
    column_count = len(df.columns)
    ingestion_time = datetime.now()

    # Add technical metadata
    df = (
        df
            .withColumn("ingestion_timestamp", current_timestamp())
            .withColumn("source_file", col("_metadata.file_path"))
    )

    # Write Bronze Delta table
    (
        df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}")
    )

    # Print ingestion summary
    print("=" * 60)
    print(f"✓ Bronze Table Created : {table_name}")
    print(f"Source File            : {source_file}")
    print(f"Rows Ingested          : {row_count:,}")
    print(f"Columns                : {column_count}")
    print(f"Ingestion Time         : {ingestion_time}")
    print("=" * 60)

    # Return statistics
    return {
        "source_file": source_file,
        "target_table": table_name,
        "rows": row_count,
        "columns": column_count,
        "ingested_at": ingestion_time
    }

In [0]:
# all at once now via the for loop
datasets = [
    ("olist_customers_dataset.csv", "customers_raw"),
    ("olist_orders_dataset.csv", "orders_raw"),
    ("olist_order_items_dataset.csv", "order_items_raw"),
    ("olist_products_dataset.csv", "products_raw"),
    ("olist_order_payments_dataset.csv", "payments_raw"),
    ("olist_order_reviews_dataset.csv", "reviews_raw"),
    ("olist_sellers_dataset.csv", "sellers_raw"),
    ("olist_geolocation_dataset.csv", "geolocation_raw"),
    ("product_category_name_translation.csv", "product_category_translation_raw")
]

for source_file, table_name in datasets:
    ingest_to_bronze(source_file, table_name)

In [0]:
reviews_df = (
    spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("multiLine", True)
        .csv(f"{BASE_PATH}/olist_order_reviews_dataset.csv")
)

In [0]:
print(f"Rows: {reviews_df.count():,}")
print(f"Columns: {len(reviews_df.columns)}")

reviews_df.show(10, truncate=False)

In [0]:
from pyspark.sql.functions import current_timestamp, col
reviews_df.groupBy("review_id") \
    .count() \
    .filter(col("count") > 1) \
    .orderBy(col("count").desc()) \
    .show(20, truncate=False)

In [0]:
reviews_df.select(
    "review_id",
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp"
).show(20, truncate=False)

In [0]:
(
    reviews_df
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.reviews_raw")
)